# 05 - Comparativa de Resultados

Comparamos los tres enfoques (lexicón, BETO, LLM) sobre el mismo conjunto de test.
Reportamos:
- Accuracy, F1 macro y micro por enfoque.
- Matriz de confusión.
- Scores de reputación agregados 0-5.
- MAE/RMSE/Pearson contra ratings reales.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se encontró la raíz del repositorio")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import load_reviews
from src.reputation.aggregator import (
    aspect_score_summary,
    compute_global_reputation,
    compute_reputation_scores,
)
from src.evaluation.metrics import evaluate_reputation

## 1. Cargar resultados

Asumimos que `scripts/eval_all.py` se ejecutó y dejó un JSON en `reports/eval_results.json`.

In [ ]:
from pathlib import Path
results_path = REPO_ROOT / "reports/eval_results.json"
if results_path.exists():
    with results_path.open(encoding="utf-8") as fh:
        results = json.load(fh)
else:
    print("Ejecuta primero: python scripts/eval_all.py --data data/sample/reviews_sample.csv --allow-pseudo-smoke")
    results = {}
list(results.keys())


## 2. Tabla comparativa

In [ ]:
rows = []
for approach in ('lexicon', 'svm', 'beto', 'llm'):
    if approach in results:
        m = results[approach]
        rows.append({
            'enfoque': approach,
            'accuracy': m['accuracy'],
            'f1_macro': m['f1_macro'],
            'f1_micro': m['f1_micro'],
        })
comparison = pd.DataFrame(rows)
comparison

In [ ]:
if comparison.empty:
    print("No hay métricas para graficar todavía.")
else:
    comparison.set_index("enfoque")[["accuracy", "f1_macro"]].plot.bar(figsize=(8, 4))
    plt.title("Comparativa de enfoques")
    plt.ylabel("score")
    plt.xticks(rotation=0)
    plt.show()


## 3. Scores de reputación agregada

In [ ]:
reputation = results.get("reputation_scores", {})
scores = reputation.get("scores", reputation) if isinstance(reputation, dict) else {}
summary = aspect_score_summary(scores)
pd.DataFrame({"score": scores, "etiqueta": summary})


In [ ]:
if scores:
    print("Reputación global:", round(compute_global_reputation(scores), 2), "/ 5")
else:
    print("No hay puntuaciones de reputación disponibles.")


## 4. Discusión

- **Lexicón:** rápido y explicable, falla con sarcasmo y negaciones complejas.
- **SVM:** estable con dataset moderado; depende de la calidad de pseudo-labels.
- **BETO:** mejor F1 esperado, requiere GPU.
- **LLM:** flexible, ideal para zero/few-shot, pero más caro y con latencia.